# Nool - Enterprise Knowledge Assistant: GPU Answer Evaluation

This notebook checks Qwen3 4B Instruct and Qwen 2.5 3B on the same 162 company questions using a Colab GPU. Both use the same hybrid search index and passage selector. The 50,000-passage search benchmark stays unchanged.

1. In Colab, choose **Runtime > Change runtime type > T4 GPU**.
2. Put `company_search_artifact.zip` in `MyDrive/Nool/`.
3. Run each cell from top to bottom. Keep the notebook open until the answer checks finish.
4. Download both prediction files from `MyDrive/Nool/` and share them for review.

Each answer is saved to Drive as it finishes. If Colab disconnects, reconnect and rerun the cells; the last cell resumes saved work. Ollama runs inside Colab, so no API key is needed. The runtime is temporary.


## Prepare the project and search index

This cell checks the GPU, connects Google Drive, gets the project code, loads the existing index ZIP, and prepares the original fictional company database. Approve the Drive access prompt when Colab asks.


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then run this cell again.")

drive.mount("/content/drive")
repo = Path("/content/Nool")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/nayanojwal0810/Nool.git", str(repo)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyyaml>=6.0.1",
        "bm25s==0.3.16",
        "PyStemmer==3.1.0",
        "pypdf>=5.0,<7.0",
        "transformers==5.22.1",
    ],
    check=True,
)

artifact = Path("/content/drive/MyDrive/Nool/company_search_artifact.zip")
if not artifact.is_file():
    raise FileNotFoundError(f"Put the existing company search ZIP at {artifact}")
with zipfile.ZipFile(artifact) as archive:
    archive.extractall(repo / "indexes")

project_env = os.environ.copy()
project_env["PYTHONPATH"] = str(repo / "src")
subprocess.run(
    [sys.executable, "scripts/ingest_demo_documents.py"],
    cwd=repo,
    env=project_env,
    check=True,
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Company search index and original demo documents are ready.")

## Start the answer models

This cell installs Ollama inside Colab and downloads Qwen3 4B Instruct and Qwen 2.5 3B. It checks that each model uses the GPU. The download can take several minutes.


In [ ]:
import time
from urllib.request import urlopen

archive_path = "/tmp/ollama-linux-amd64.tar.zst"
subprocess.run(
    [
        "curl",
        "-fL",
        "-o",
        archive_path,
        "https://ollama.com/download/ollama-linux-amd64.tar.zst",
    ],
    check=True,
)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
subprocess.run(["tar", "-C", "/usr", "-xf", archive_path], check=True)
log = open("/content/ollama_server.log", "w", encoding="utf-8")
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
)
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError("Ollama stopped. Check /content/ollama_server.log")
    try:
        with urlopen("http://127.0.0.1:11434/api/tags", timeout=2):
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Check /content/ollama_server.log")
for model_name in ("qwen3:4b-instruct", "qwen2.5:3b"):
    subprocess.run(["ollama", "pull", model_name], check=True)
    subprocess.run(
        ["ollama", "run", model_name, "Reply OK only."],
        check=True,
        stdout=subprocess.PIPE,
        text=True,
    )
    placement = subprocess.check_output(["ollama", "ps"], text=True)
    model_row = next(
        (row for row in placement.splitlines() if row.startswith(model_name)), ""
    )
    if "100% GPU" not in model_row:
        raise RuntimeError(
            f"{model_name} is using CPU. Select a GPU runtime and restart."
        )
    print(f"{model_name} is ready on GPU.")
    subprocess.run(["ollama", "stop", model_name], check=True)

## Check all prepared answers

This cell runs both models on every initial-state case with the same passages and timeout. Qwen3 runs first. It prints automatic counts and saves each answer to Drive. The counts are not an answer-accuracy score.


In [ ]:
for model_name, filename in (
    ("qwen3:4b-instruct", "company_answer_predictions_qwen3.jsonl"),
    ("qwen2.5:3b", "company_answer_predictions_qwen25.jsonl"),
):
    output = Path("/content/drive/MyDrive/Nool") / filename
    try:
        subprocess.run(
            [
                sys.executable,
                "scripts/evaluate_company_answers.py",
                "--model",
                model_name,
                "--timeout",
                "240",
                "--resume",
                "--device",
                "cuda",
                "--output",
                str(output),
            ],
            cwd=repo,
            env=project_env,
            check=True,
        )
        print(f"Saved review file: {output}")
    finally:
        subprocess.run(["ollama", "stop", model_name], check=False)

## Review the results

Download `company_answer_predictions_qwen3.jsonl` and `company_answer_predictions_qwen25.jsonl` from Drive and share them for review. Check factual accuracy, whether each answer follows from its cited excerpt, two-source coverage, and refusals. The four document-update and four permission-change cases are tested separately in the project tests.
